# Arsitrad — Gemma 4 Inference Demo
## End-to-End: RAG + Fine-tuned Gemma 4 2B → Gradio UI

**Runtime:** Colab with **2x T4 GPU** (High-RAM recommended)

---


## 1. Setup & Installation


In [2]:
#@title 1.2 — Install dependencies
!pip install -q --upgrade pip
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes \
    accelerate \
    scipy \
    chromadb>=0.4.0 \
    sentence-transformers>=2.5.0 \
    gradio>=4.0.0 \
    huggingface_hub \
    tqdm
!pip install -q unsloth
print('All dependencies installed!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exp

In [3]:
#@title 1.3 — Imports & Config
import os
import torch
from pathlib import Path

BASE_DIR   = Path('/content/arsitrad')
LORA_DIR   = BASE_DIR / 'lora_adapters'
CHROMA_DIR = BASE_DIR / 'chroma_db'

RAG_CONFIG = {
    'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'collection_name': 'arsitrad_national_regulations',
    'top_k': 5,
    'min_similarity': 0.3,
}

MODEL_CONFIG = {
    'max_seq_length': 2048,
    'load_in_4bit': True,
    'device_map': 'auto',                       # shards base model across all GPUs
    'base_model': 'google/gemma-4-2b-it',      # full base model for device_map="auto"
}
print('Config ready.')


Config ready.


## 2. Download LoRA Adapters + ChromaDB


In [4]:
#@title 2.1 — Create directories
!mkdir -p /content/arsitrad/lora_adapters
!mkdir -p /content/arsitrad/chroma_db
print('Directories created.')


Directories created.


In [5]:
#@title 2.2 — Download LoRA adapters from GitHub Release
import os, subprocess

LORA_URL = 'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/arsitrad_finetuned_adapters.zip'
out = '/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters.zip'

if os.path.exists('/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters'):
    print('LoRA adapters already exist, skipping download.')
else:
    print('Downloading LoRA adapters (~747 MB)...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', out, LORA_URL], check=True)
    print('Unzipping...')
    subprocess.run(['unzip', '-q', '-o', out, '-d', '/content/arsitrad/lora_adapters/'], check=True)
    print('Done!')


Unzipping...
Done!


In [13]:
#@title 2.3 — Download and Fix ChromaDB Extraction
import os, subprocess, shutil

CHROMA_ZIP = '/content/arsitrad/chroma_db_release.zip'
TARGET_DIR = '/content/arsitrad/chroma_db'
TEMP_EXTRACT = '/content/arsitrad/temp_chroma'

# Clean up previous attempts
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
if os.path.exists(TEMP_EXTRACT): shutil.rmtree(TEMP_EXTRACT)
os.makedirs(TARGET_DIR, exist_ok=True)

print('Downloading ChromaDB (~262 MB)...')
subprocess.run(['wget', '-q', '--show-progress', '-O', CHROMA_ZIP, 'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/chroma_db_release.zip'], check=True)

print('Extracting to temporary folder...')
subprocess.run(['unzip', '-q', '-o', CHROMA_ZIP, '-d', TEMP_EXTRACT], check=True)

# Find where the actual sqlite file is (handling nested folders)
sqlite_source_dir = None
for root, dirs, files in os.walk(TEMP_EXTRACT):
    if 'chroma.sqlite3' in files:
        sqlite_source_dir = root
        break

if sqlite_source_dir:
    print(f'Found database in {sqlite_source_dir}. Moving to {TARGET_DIR}...')
    for item in os.listdir(sqlite_source_dir):
        shutil.move(os.path.join(sqlite_source_dir, item), os.path.join(TARGET_DIR, item))
    print('Done!')
else:
    print('Error: chroma.sqlite3 not found in the downloaded ZIP.')

# Cleanup
shutil.rmtree(TEMP_EXTRACT)
if os.path.exists(CHROMA_ZIP): os.remove(CHROMA_ZIP)
print('Final Contents of', TARGET_DIR, ':', os.listdir(TARGET_DIR))

Extracting to temporary folder...
Found database in /content/arsitrad/temp_chroma/chroma_db. Moving to /content/arsitrad/chroma_db...
Done!
Final Contents of /content/arsitrad/chroma_db : ['chroma.sqlite3', 'f74d0699-6baf-4875-ae19-9ad6551fd1d7', 'b3df3f29-b6f3-4249-bc84-194a64429fd7', 'national_embeddings.npy', '0a9eb16f-cb59-4afd-bbc3-d5e2695b65dd', 'national_chunks.json']


## 3. RAG Retrieval Pipeline


In [21]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
import importlib

# Force a clean state at the library level
importlib.reload(chromadb)
from chromadb.api.shared_system_client import SharedSystemClient
SharedSystemClient.clear_system_cache()

class RegulationRetriever:
    def __init__(self, persist_dir, collection_name):
        self.emb_model = SentenceTransformer(RAG_CONFIG['embedding_model'])
        # Using standard settings to avoid conflict with existing cached systems
        self.client = chromadb.PersistentClient(path=persist_dir)

        try:
            # Try to get existing collection
            self.collection = self.client.get_collection(name=collection_name)
            print(f'✅ Success! Collection "{collection_name}" loaded with {self.collection.count()} chunks.')
        except Exception as e:
            print(f'❌ Failed to get collection: {e}')
            cols = self.client.list_collections()
            print('Available collections:', [c.name for c in cols] if cols else 'None')
            raise e

    def retrieve(self, query, top_k=5, min_similarity=0.3):
        raw_emb = self.emb_model.encode([query])[0]
        q_norm = raw_emb / (np.linalg.norm(raw_emb) + 1e-8)
        results = self.collection.query(
            query_embeddings=[q_norm.tolist()],
            n_results=top_k,
            include=['documents','metadatas','embeddings'],
        )
        outputs = []
        for doc, metadata, emb in zip(results['documents'][0],
                                      results['metadatas'][0],
                                      results['embeddings'][0]):
            if emb is None: continue
            cos_sim = float(np.dot(q_norm, np.array(emb)) / (np.linalg.norm(np.array(emb)) + 1e-8))
            if cos_sim >= min_similarity:
                outputs.append({
                    'text': doc,
                    'source': metadata.get('source','unknown'),
                    'chunk_id': metadata.get('chunk_id',0),
                    'similarity': round(cos_sim,4)
                })
        outputs.sort(key=lambda x: x['similarity'], reverse=True)
        return outputs

    def format_context(self, retrieved):
        if not retrieved:
            return 'No relevant regulations found.'
        parts = []
        for i, item in enumerate(retrieved, 1):
            parts.append(f"[{i}] {item['text']}\n(Source: {item['source']}, chunk {item['chunk_id']})")
        return '\n\n'.join(parts)

# Re-initialize after clearing cache
retriever = RegulationRetriever(str(CHROMA_DIR), RAG_CONFIG['collection_name'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Success! Collection "arsitrad_national_regulations" loaded with 22649 chunks.


In [15]:
import sqlite3

db_path = '/content/arsitrad/chroma_db/chroma.sqlite3'
conn = sqlite3.connect(db_path)
cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables:", [r[0] for r in cursor.fetchall()])

try:
    cursor2 = conn.execute("SELECT id, name FROM collections;")
    print("Collections:", cursor2.fetchall())
except Exception as e:
    print(f"Error: {e}")
conn.close()

Tables: ['migrations', 'acquire_write', 'collection_metadata', 'segment_metadata', 'tenants', 'databases', 'collections', 'maintenance_log', 'segments', 'embeddings', 'embedding_metadata', 'max_seq_id', 'embedding_fulltext_search', 'embedding_fulltext_search_data', 'embedding_fulltext_search_idx', 'embedding_fulltext_search_content', 'embedding_fulltext_search_docsize', 'embedding_fulltext_search_config', 'embedding_metadata_array', 'embeddings_queue', 'embeddings_queue_config']
Collections: [('3ee3c606-2946-48ee-acea-cf7e9dc8dfa0', 'arsitrad_national_regulations'), ('bab40dd5-19e0-43b9-8ac8-136a3e74bcb3', 'arsitrad_local_regulations')]


In [16]:
!pip install -q --upgrade chromadb
import chromadb
print(f"Updated ChromaDB to version: {chromadb.__version__}")

Updated ChromaDB to version: 1.5.7


In [17]:
import chromadb
import importlib
importlib.reload(chromadb)

# Initialize client with the absolute path
client = chromadb.PersistentClient(path="/content/arsitrad/chroma_db")

try:
    # Attempt to fetch the specific collection
    coll = client.get_collection("arsitrad_national_regulations")
    print(f"✅ Success! Collection 'arsitrad_national_regulations' found.")
    print(f"📊 Total chunks loaded: {coll.count()}")
except Exception as e:
    print(f"❌ Error retrieving collection: {e}")
    print("\nAvailable collections in this database:")
    collections = client.list_collections()
    if not collections:
        print("  (No collections found)")
    for c in collections:
        print(f"  - {c.name}")

❌ Error retrieving collection: Collection [arsitrad_national_regulations] does not exist

Available collections in this database:
  (No collections found)


In [18]:
import os

# Check what's actually in the chroma_db directory
if os.path.exists('/content/arsitrad/chroma_db/'):
    print("Contents of /content/arsitrad/chroma_db/:")
    for f in os.listdir('/content/arsitrad/chroma_db/'):
        file_path = f'/content/arsitrad/chroma_db/{f}'
        size = os.path.getsize(file_path)
        print(f"  {f} — {size/1e6:.1f} MB")
else:
    print("Directory /content/arsitrad/chroma_db/ does not exist.")

Contents of /content/arsitrad/chroma_db/:
  chroma.sqlite3 — 260.4 MB
  f74d0699-6baf-4875-ae19-9ad6551fd1d7 — 0.0 MB
  b3df3f29-b6f3-4249-bc84-194a64429fd7 — 0.0 MB
  national_embeddings.npy — 34.8 MB
  0a9eb16f-cb59-4afd-bbc3-d5e2695b65dd — 0.0 MB
  national_chunks.json — 18.4 MB


In [19]:
import sqlite3
import os

db_path = '/content/arsitrad/chroma_db/chroma.sqlite3'
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [r[0] for r in cursor.fetchall()]
    print("Tables in ChromaDB:", tables)

    if 'collections' in tables:
        cursor2 = conn.execute("SELECT id, name FROM collections;")
        rows = cursor2.fetchall()
        print(f"Collections found in SQLite: {rows}")
    else:
        print("CRITICAL: 'collections' table missing from SQLite file.")
    conn.close()
else:
    print(f"SQLite file not found at {db_path}")

Tables in ChromaDB: ['migrations', 'acquire_write', 'collection_metadata', 'segment_metadata', 'tenants', 'databases', 'collections', 'maintenance_log', 'segments', 'embeddings', 'embedding_metadata', 'max_seq_id', 'embedding_fulltext_search', 'embedding_fulltext_search_data', 'embedding_fulltext_search_idx', 'embedding_fulltext_search_content', 'embedding_fulltext_search_docsize', 'embedding_fulltext_search_config', 'embedding_metadata_array', 'embeddings_queue', 'embeddings_queue_config']
Collections found in SQLite: [('3ee3c606-2946-48ee-acea-cf7e9dc8dfa0', 'arsitrad_national_regulations'), ('bab40dd5-19e0-43b9-8ac8-136a3e74bcb3', 'arsitrad_local_regulations')]


In [22]:
#@title 3.2 — Test RAG retrieval
test_query = 'syarat teknis ventilasi alami pada rumah tinggal'
results   = retriever.retrieve(test_query, top_k=3)

print(f'Query: {test_query}\n')
for r in results:
    print(f"  [{r['similarity']:.3f}] {r['source']} — {r['text'][:120]}...")
print(f'\nFormatted context:\n{retriever.format_context(results)[:400]}')


Query: syarat teknis ventilasi alami pada rumah tinggal

  [0.828] Permen_21_2021_BGH —  segar ke 
dalam Bangunan Gedung dalam jumlah yang sesuai 
kebutuhan. Selain untuk menyirkulasi gas-gas yang 
berbahaya,...
  [0.680] Permen_26_2008_ProteksiKebakaran — rus sesuai dengan persyaratan teknis untuk 
peralatan memasak komersial, kecuali instalasi tersebut instalasi yang sudah...
  [0.401] SNI_1727_2020_BebanDesainMinimumDanKriteriaTerikat — ondasi
Faktor ketahanan 
harus diberi nilai 0,67 yang diterapkan pada kapasitas ketahanan
untuk penggunaan dengan analis...

Formatted context:
[1]  segar ke 
dalam Bangunan Gedung dalam jumlah yang sesuai 
kebutuhan. Selain untuk menyirkulasi gas-gas yang 
berbahaya, Ventilasi juga digunakan semaksimal mungkin 
untuk meminimalkan beban pendinginan. Sistem ventilasi 
terbagi menjadi dua yaitu sistem ventilasi mekanis dan 
sistem ventilasi alami. Sistem ventilasi mekanis harus 
disediakan apabila sistem ventilasi alami tidak memadai. 
Pere


## 4. Load Fine-tuned Gemma 4 + Inference


In [23]:
#@title 4.1 — Load Gemma 4 2B + LoRA (DEPRECATED — use merged model below)
# CRITICAL: import unsloth FIRST, before any other imports.
# Unsloth patches torch.nn.Linear -> Gemma4ClippableLinear globally.
# peft must see the patched layers when it injects LoRA adapters.

import unsloth  # ← MUST be first! Triggers global layer patching.
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel

BASE_MODEL = "google/gemma-4-e2b-it"
LORA_PATH  = "/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters"

# Step 1: Write adapter_config.json if missing
import json, os
adapter_config = {
    "base_model_name_or_path": BASE_MODEL,
    "bias": "none",
    "inference_mode": True,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "modules_to_save": None,
    "peft_type": "LORA",
    "r": 64,
    "target_modules": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    "task_type": "CAUSAL_LM",
}
os.makedirs(LORA_PATH, exist_ok=True)
if not os.path.exists(os.path.join(LORA_PATH, "adapter_config.json")):
    with open(os.path.join(LORA_PATH, "adapter_config.json"), "w") as f:
        json.dump(adapter_config, f, indent=2)
    print("Created adapter_config.json")
else:
    print("adapter_config.json already exists")

# Step 2: Load base model with Unsloth (import order matters!)
total_mem = {i: int(torch.cuda.get_device_properties(i).total_memory)
             for i in range(torch.cuda.device_count())}
print(f"GPUs: {total_mem}")

print(f"Loading {BASE_MODEL} with Unsloth (4-bit)...")
model, _ = FastLanguageModel.from_pretrained(
    model_name             = BASE_MODEL,
    max_seq_length         = 2048,
    load_in_4bit           = True,
    dtype                  = None,
    device_map             = "auto",
    max_memory             = total_mem,
    gpu_memory_utilization = 0.7,
)
print("Base model loaded.")

# Step 3: Attach LoRA adapters — peft must use the already-patched layer types
print("Attaching LoRA adapters...")
model = PeftModel.from_pretrained(model, LORA_PATH)
FastLanguageModel.for_inference(model)

# Step 4: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Verify GPUs
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")

print(f"\n✅ Model ready! LoRA: {LORA_PATH}")

# ⚠️ OBSOLETE: Run cell 4.2 ONCE to create merged model, then replace this cell with:
# model, tokenizer = FastLanguageModel.from_pretrained(HF_REPO, max_seq_length=2048, load_in_4bit=True)


/tmp/ipykernel_3089/2061001445.py:6: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Model + tokenizer loaded!


In [ ]:
#@title 4.2 — Merge LoRA → Push to HuggingFace (run once, then delete this cell)
# Produces a self-contained merged model. After this runs, cell 4.1 becomes obsolete.
# Requirements: internet ON + HF token logged in.

import unsloth, torch
from unsloth import FastLanguageModel
from peft import PeftModel
from huggingface_hub import HfApi, login

HF_REPO = "YOUR_HF_USERNAME/arsitrad-gemma4-2b-it"  # ← change this

print("Logging in to HuggingFace...")
from huggingface_hub import notebook_login
notebook_login()

print("Loading base model...")
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name             = BASE_MODEL,
    max_seq_length         = 2048,
    load_in_4bit           = True,
    dtype                  = None,
    device_map             = "auto",
    max_memory             = {i: int(torch.cuda.get_device_properties(i).total_memory * 0.75)
                              for i in range(torch.cuda.device_count())},
)

print("Attaching LoRA...")
model = PeftModel.from_pretrained(base_model, LORA_PATH)

print("Merging LoRA into base weights (takes a few minutes)...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./arsitrad_merged")
tokenizer.save_pretrained("./arsitrad_merged")
print("Merged model saved to ./arsitrad_merged")

print(f"Pushing to HuggingFace → {HF_REPO}...")
api = HfApi()
api.create_repo(repo_id=HF_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="./arsitrad_merged", repo_id=HF_REPO, repo_type="model")
print(f"✅ Pushed! Update cell 4.1 to load from:\n"
      f'   model_name = "{HF_REPO}"')


In [24]:
#@title 4.2 — Build prompt with RAG context (with optional reasoning mode)
SYSTEM_PROMPT = (
    'Kamu adalah Arsitrad, asisten AI berlisensi untuk regulasi dan arsitektur Indonesia. '
    'Jawab secara komprehensif — gunakan konteks peraturan yang diberikan untuk memberikan '
    'jawaban yang detail, terstruktur, dan akurat. '
    'Sertakan rujukan spesifik (nama peraturan dan pasalnya) bila memungkinkan. '
    'Jika konteks tidak cukup, jawablah dengan pengetahuan Anda secara jujur dan jelaskan asumsinya.'
)

def build_prompt(user_query, context, use_reasoning=False):
    if context and context != 'No relevant regulations found.':
        full_query = (
            'Konteks peraturan (jawablah MERujuk ke konteks ini):\n'
            + context + '\n\n'
            'Pertanyaan pengguna:\n' + user_query
        )
    else:
        full_query = user_query

    prompt = (
        '<start_of_turn>system\n' + SYSTEM_PROMPT + '<end_of_turn>\n'
        '<start_of_turn>user\n' + full_query + '<end_of_turn>\n'
    )
    if use_reasoning:
        prompt += '<start_of_turn>thought\n'
    else:
        prompt += '<start_of_turn>model\n'
    return prompt


In [35]:
#@title 4.3 — Streaming generator (Gemma 4 reasoning mode support)
from transformers import TextIteratorStreamer
from threading import Thread

def generate_response_stream(user_query,
                              max_new_tokens=512,
                              temperature=0.2,
                              top_p=0.95,
                              top_k=64,
                              repetition_penalty=1.05,
                              use_reasoning=False):
    retrieved  = retriever.retrieve(user_query, top_k=RAG_CONFIG['top_k'])
    context    = retriever.format_context(retrieved)
    prompt     = build_prompt(user_query, context, use_reasoning=use_reasoning)
    inputs     = tokenizer(prompt, return_tensors='pt', truncation=True)
    # Use first GPU for inputs — device_map='balanced' distributes model across GPUs
    device = torch.device('cuda:0')
    inputs     = {k: v.to(device) for k, v in inputs.items()}

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=repetition_penalty,
        do_sample=True,
        streamer=streamer,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    accumulated  = ''
    thought_block = ''

    for new_text in streamer:
        accumulated += new_text

        if use_reasoning:
            t_start_tag = '<start_of_turn>thought\n'
            if t_start_tag in accumulated:
                t_start = accumulated.find(t_start_tag) + len(t_start_tag)
                t_end = accumulated.find('<end_of_turn>', t_start)
                if t_end != -1:
                    thought_block = accumulated[t_start:t_end]
                    after_thought = accumulated[t_end + len('<end_of_turn>'):]
                    final_answer = after_thought.replace('<start_of_turn>model\n', '', 1)
                    yield final_answer, context, retrieved, thought_block
                else:
                    yield '', context, retrieved, ''
            else:
                yield '', context, retrieved, ''
        else:
            yield accumulated, context, retrieved, ''

    thread.join()


In [36]:
#@title 4.4 — Test inference with streaming + reasoning mode demo
import time

test_queries = [
    'Apa saja syarat IMB untuk rumah tinggal 2 lantai di Semarang?',
    'Bagaimana klasifikasi kerusakan bangunan akibat gempa bumi?',
    'Strategi passive cooling untuk rumah di daerah tropis basah?',
]

# Normal generation (reasoning=off)
for q in test_queries:
    print(f'\n{"="*60}')
    print(f'Q: {q}')
    print(f'{"-"*60}')
    print('R: ', end='', flush=True)

    accumulated = ''
    start = time.time()
    for partial_response, _, retrieved, _ in generate_response_stream(q, max_new_tokens=512, use_reasoning=False):
        new_tokens = partial_response[len(accumulated):]
        print(new_tokens, end='', flush=True)
        accumulated = partial_response
    elapsed = time.time() - start
    print(f'\n\n[time {elapsed:.1f}s | {len(accumulated)} chars | {len(retrieved)} chunks cited | reasoning=off]')

# Reasoning mode demo — first query only
print(f'\n\n{"="*60}')
print('REASONING MODE DEMO — first query')
print(f'{"="*60}')
q = test_queries[0]
print(f'Q: {q}')
print('R: ', end='', flush=True)

thought_acc = ''
accumulated = ''
start = time.time()
for partial_response, _, retrieved, thought in generate_response_stream(q, max_new_tokens=512, use_reasoning=True):
    new_tokens = partial_response[len(accumulated):]
    print(new_tokens, end='', flush=True)
    accumulated = partial_response
    if thought:
        thought_acc = thought
elapsed = time.time() - start

print(f'\n\n[time {elapsed:.1f}s | {len(accumulated)} chars]')
if thought_acc:
    print(f'[thought preview: {thought_acc[:300]}...]')

print('\nAll test queries passed!')



Q: Apa saja syarat IMB untuk rumah tinggal 2 lantai di Semarang?
------------------------------------------------------------
R: <start_of_turn

[Cited 0 regulation chunks]

Q: Bagaimana klasifikasi kerusakan bangunan akibat gempa bumi?
------------------------------------------------------------
R: Klasifikasi kerusakan bangunan akibat gempa bumi dapat diklasifikasikan berdasarkan beberapa aspek utama, yang mencakup:

1. **Kerusakan Struktural (Structural Damage):**
   * **Kerusakan Non-Struktural:** Melibatkan kerusakan pada elemen-elemen pendukung, seperti kolom, balok, dan dinding, yang menyebabkan perubahan pada geometri dan distribusi beban. Kerusakan ini dapat bervariasi dari retak permukaan hingga kegagalan total.
   * **Kerusakan Geometris:** Melibatkan perubahan bentuk fisik bangunan akibat gaya gempa, yang dapat menyebabkan deformasi yang signifikan.
   * **Kerusakan Distribusi

[Cited 5 regulation chunks]

Q: Strategi passive cooling untuk rumah di daerah tropis basah?
---

In [34]:
# Force fastest inference mode
import torch

# Disable gradient tracking globally – not needed for inference
torch.set_grad_enabled(False)

# Verify model is in eval/ inference mode
model.eval()
print(f"Model training mode: {model.training}")
print(f"Inference optimizations applied. Please re-run the following cells.")

Model training mode: False
Inference optimizations applied. Please re-run the following cells.


## 5. Gradio UI — Live Demo


In [28]:
#@title 5.1 — Gradio UI: streaming + all gen params + reasoning toggleimport gradio as grimport timecustom_css = """:root {    --cc-primary: #3b82f6;    --cc-primary-hover: #2563eb;    --cc-bg: #0a0a0a;    --cc-surface: #171717;    --cc-surface-2: #1f1f1f;    --cc-border: #262626;    --cc-text: #ededed;    --cc-muted: #a3a3a3;}body, html { margin: 0; padding: 0; background: var(--cc-bg); font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; color: var(--cc-text); }.header-bar { position: sticky; top: 0; z-index: 100; display: flex; align-items: center; justify-content: space-between; padding: 0 20px; height: 52px; background: rgba(10,10,10,0.85); backdrop-filter: blur(20px); -webkit-backdrop-filter: blur(20px); border-bottom: 1px solid var(--cc-border); }.header-logo { font-size: 16px; font-weight: 700; background: linear-gradient(135deg, #3b82f6, #8b5cf6); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }.header-sub { font-size: 11px; color: var(--cc-muted); }.header-badge { display: inline-flex; align-items: center; gap: 5px; background: rgba(59,130,246,0.12); border: 1px solid rgba(59,130,246,0.25); color: #60a5fa; font-size: 11px; font-weight: 500; padding: 3px 10px; border-radius: 99px; }.header-badge::before { content: ""; width: 6px; height: 6px; background: #3b82f6; border-radius: 50%; box-shadow: 0 0 6px #3b82f6; }#chat-wrap { max-width: 780px; margin: 0 auto; padding: 0 16px; }.message.user { background: var(--cc-primary) !important; color: white !important; border-radius: 14px 14px 4px 14px !important; padding: 10px 14px !important; font-size: 14px !important; max-width: 85% !important; margin-left: auto !important; box-shadow: 0 4px 20px rgba(0,0,0,0.4) !important; border: none !important; }.message.assistant { background: var(--cc-surface-2) !important; color: var(--cc-text) !important; border-radius: 14px 14px 14px 4px !important; padding: 10px 14px !important; font-size: 14px !important; max-width: 85% !important; border: 1px solid var(--cc-border) !important; }.sources-box { margin-top: 8px; padding: 8px 12px; background: rgba(59,130,246,0.08); border: 1px solid rgba(59,130,246,0.2); border-radius: 8px; font-size: 12px; color: #9ca3af; }.sources-title { font-weight: 600; color: #60a5fa; font-size: 11px; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 4px; }.source-item { margin: 2px 0; }.input-wrap { position: sticky; bottom: 0; z-index: 50; padding: 12px 16px 16px; background: linear-gradient(to top, var(--cc-bg) 80%, transparent); }.input-row { display: flex; align-items: flex-end; gap: 10px; background: var(--cc-surface); border: 1px solid var(--cc-border); border-radius: 14px; padding: 10px 14px; transition: border-color 200ms; }.input-row:focus-within { border-color: var(--cc-primary); box-shadow: 0 0 0 3px rgba(59,130,246,0.12); }textarea { background: transparent !important; border: none !important; outline: none !important; color: var(--cc-text) !important; font-size: 14px !important; line-height: 1.5 !important; resize: none !important; width: 100% !important; }textarea::placeholder { color: #525252; }.send-btn { background: var(--cc-primary) !important; border: none !important; border-radius: 8px !important; color: white !important; font-size: 13px !important; font-weight: 600 !important; padding: 8px 18px !important; cursor: pointer; transition: background 150ms, transform 100ms; white-space: nowrap; height: fit-content; }.send-btn:hover { background: var(--cc-primary-hover) !important; }.send-btn:active { transform: scale(0.97); }.examples-bar { display: flex; flex-wrap: wrap; gap: 6px; padding: 10px 0 6px; border-bottom: 1px solid var(--cc-border); margin-bottom: 6px; }.example-chip { background: var(--cc-surface-2); border: 1px solid var(--cc-border); color: #a3a3a3; font-size: 12px; padding: 4px 12px; border-radius: 99px; cursor: pointer; transition: all 150ms; }.example-chip:hover { background: var(--cc-surface); border-color: var(--cc-primary); color: #60a5fa; }.loading-indicator { display: inline-flex; gap: 4px; align-items: center; }.loading-indicator span { width: 5px; height: 5px; background: var(--cc-muted); border-radius: 50%; animation: dot-pulse 1.2s infinite; }.loading-indicator span:nth-child(2) { animation-delay: 0.2s; }.loading-indicator span:nth-child(3) { animation-delay: 0.4s; }@keyframes dot-pulse { 0%,80%,100% { opacity: 0.3; transform: scale(0.8); } 40% { opacity: 1; transform: scale(1); } }.slider-label { color: #a3a3a3; font-size: 12px; }::-webkit-scrollbar { width: 5px; }::-webkit-scrollbar-track { background: transparent; }::-webkit-scrollbar-thumb { background: #333; border-radius: 3px; }"""HEADER_HTML = """<div class="header-bar">    <div>        <div class="header-logo">ARSITRAD</div>        <div class="header-sub">AI Konsultan Regulasi &amp; Arsitektur Indonesia</div>    </div>    <div class="header-badge">Gemma 4 &middot; RAG &middot; 2x T4 &middot; Reasoning Mode</div></div>"""EXAMPLES = [    'Rumah rusak akibat gempa Cianjur, dinding retak diagonal 45&deg;, atap bergeser. Apa yang harus dilakukan?',    'Apa syarat memperoleh Izin Mendirikan Bangunan (IMB) di Indonesia?',    'Jelaskan strategi passive cooling untuk bangunan tropis Indonesia.',    'Bagaimana prosedur upgrading permukiman kumuh sesuai regulasi?',]CHIPS_HTML = '<div class="examples-bar">' + ''.join([    '<div class="example-chip" onclick="var t=document.querySelector(\'textarea\');t.value=\'"'"'' + e.replace('&deg;', '\u00b0').replace("'", "\u0027") + '\'"'"';t.dispatchEvent(new Event(\'input\'));document.querySelector(\'.send-btn\').click()">' + e[:55] + '...</div>'    for e in EXAMPLES]) + '</div>'def arsitrad_stream(message, history, max_tokens, temperature, top_p, top_k, use_reasoning):    """    Streaming inference with optional Gemma 4 reasoning mode.    use_reasoning=True: model first shows its internal reasoning steps,    then delivers the final answer.    """    for partial_response, _, retrieved, thought in generate_response_stream(        message,        max_new_tokens=int(max_tokens),        temperature=float(temperature),        top_p=float(top_p),        top_k=int(top_k),        use_reasoning=bool(use_reasoning),    ):        sources_html = (            '<div class="sources-box"><div class="sources-title">Sources</div>'            '<br>'.join([                f'<div class="source-item">[<span style="color:#60a5fa">{r["similarity"]:.3f}</span>] '                f'<span style="color:#d1d5db">{r["source"]}</span></div>'                for r in retrieved            ])            '</div>'        )        if thought:            thought_html = (                '<div class="sources-box" style="border-color:rgba(139,92,246,0.35);background:rgba(139,92,246,0.07);margin-bottom:6px;">'                '<div class="sources-title" style="color:#a78bfa;">[ Reasoning Steps ]</div>'                '<div style="color:#c4b5fd;font-size:12px;line-height:1.6;white-space:pre-wrap;">'                + thought.strip() + '</div></div>'            )        else:            thought_html = ''        yield (partial_response if partial_response else '') + thought_html + sources_htmlwith gr.Blocks(css=custom_css, title="Arsitrad") as demo:    gr.HTML(HEADER_HTML)    with gr.Column(elem_id="chat-wrap"):        gr.HTML(CHIPS_HTML)        chatbot = gr.Chatbot(height=420, show_copy_button=True, render=False,                             type="messages", bubble_full_width=False)        with gr.Group(elem_classes="input-wrap"):            with gr.Row(elem_classes="input-row"):                msg_input = gr.Textbox(                    placeholder="Tanyakan tentang regulasi bangunan di Indonesia...",                    lines=1, max_lines=5, show_label=False, container=False                )                send_btn = gr.Button("Kirim", elem_classes="send-btn", variant="primary")        with gr.Row():            max_tok_slider = gr.Slider(                minimum=128, maximum=1024, value=512, step=64,                label="Max Tokens  (128=singkat | 1024=sangat detail)",                scale=2,            )            temp_slider = gr.Slider(                minimum=0.0, maximum=0.7, value=0.2, step=0.05,                label="Temperature  (0=tepat | 0.2=natural | 0.5+=kreatif)",                scale=1,            )        with gr.Row():            top_p_slider = gr.Slider(                minimum=0.5, maximum=1.0, value=0.95, step=0.01,                label="Top-P  (0.8=fokus | 0.95=natural | 1.0=semua)",                scale=1,            )            top_k_slider = gr.Slider(                minimum=0, maximum=256, value=64, step=8,                label="Top-K  (0=off | 32=fokus | 64=natural | 256=kreatif)",                scale=1,            )        # Reasoning mode toggle — when ON, model shows its thinking first        with gr.Row():            reasoning_toggle = gr.Checkbox(                value=False,                label="[ Reasoning Mode ]  model explains its thinking before answering",            )        msg_input.submit(            arsitrad_stream,            [msg_input, chatbot, max_tok_slider, temp_slider, top_p_slider, top_k_slider, reasoning_toggle],            msg_input,        )        send_btn.click(            arsitrad_stream,            [msg_input, chatbot, max_tok_slider, temp_slider, top_p_slider, top_k_slider, reasoning_toggle],            msg_input,        )print("Ready — streaming + reasoning mode + all generation sliders.")demo.launch(server_name="0.0.0.0", server_port=7860, share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c31c01143e56277196.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Troubleshooting

| Problem | Fix |
|---|---|
| OOM on T4 | Runtime → Restart runtime before inference (clears GPU memory from downloads) |
| Model not loading | Use LoRA path directly, or `google/gemma-4-E2B-it` for base model |
| ChromaDB empty | Re-run Section 2.3 — wget may timeout |
| Slow retrieval | Reduce `top_k` in RAG_CONFIG (e.g. 3 instead of 5) |
| Gradio link not working | Wait 30s after launch for Colab to provision the URL |
